# Matching géométrique d’un SLM sur un banc simulé

Banc : **source monochromatique → pupille → DM de référence → SLM de phase → caméra interférométrique en plan pupille**, avec une branche **lentille → caméra focale**.

Nous mesurons des pokes push-pull du DM, ajustons la transformation affine puis encodons une OPD sur le SLM. La vérité géométrique sert uniquement au simulateur et aux vérifications, jamais aux centroïdes fournis au fit.

La caméra utilise une référence cohérente avec quatre déphasages : une caméra pupille ordinaire serait insensible à une phase seule. Le relais pupille est un modèle affine idéal, sans diffraction ni distorsion non affine. Le SLM est un adaptateur simulé construit sur `SLMEncoder`, pas un nouveau composant matériel du cœur.

Depuis la racine du dépôt :
```bash
python -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[tutorials]'
jupyter lab tutorials/15_slm_matching_simulated_bench.ipynb
```
Exécuter toutes les cellules dans l’ordre. CPU, aucune donnée externe ni SDK matériel.


In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if not (root / "fiatlux").is_dir() and (root.parent / "fiatlux").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from fiatlux import (Grid, Spectrum, PhotometricBand, PlaneWave, CircularAperture,
    Field, Detector, FFTPropagator, ActuatorGrid, SquareZonalBasis, DeformableMirror,
    AffineRegistration, SLMCalibration, SLMEncoder, select_registration_pokes,
    square_zonal_centers, weighted_centroid)
torch.set_num_threads(2)
torch.set_grad_enabled(False)
dtype = torch.float64
rng = torch.Generator().manual_seed(2026)
def show_map(ax, data, title, unit, **kwargs):
    image = ax.imshow(data.detach().cpu().numpy(), origin="lower", **kwargs)
    ax.set(title=title, xlabel="colonne", ylabel="ligne")
    plt.colorbar(image, ax=ax, label=unit)

## 1. Éléments FIATLUX et géométrie cachée du relais
Les coordonnées DM sont en mètres ; les coordonnées SLM et caméra sont des `(u, v)` en pixels, avec origine au pixel supérieur gauche dans les tableaux (les figures utilisent `origin="lower"`).

La correspondance SLM→caméra est ici une calibration préalable connue, par exemple mesurée avec des repères du panneau. Sans elle, un fit DM→caméra ne suffit pas à adresser le SLM. Le fit affine retrouve six coefficients ; au moins trois points non colinéaires sont nécessaires.


In [ ]:
grid = Grid(129, 129, 40e-6, 40e-6, dtype=dtype)
spectrum = Spectrum(magnitude=0, band=PhotometricBand.R, samples=1, dtype=dtype)
wavelength = float(spectrum.wavelengths[0])
source = PlaneWave(spectrum)
pupil = CircularAperture(grid, radius=2e-3)
illuminated = pupil.apply(source.generate_field(grid))
actuators = ActuatorGrid(9, 9, pitch=400e-6)
# SquareZonalBasis.influence_width is an absolute half-width in metres.
basis = SquareZonalBasis(actuators, grid, influence_width=150e-6)
dm = DeformableMirror(grid, actuators, grid, basis, stroke=1e-6)
slm_shape, camera_shape = (192, 224), (208, 240)
truth = AffineRegistration(
    torch.tensor([[36000., 7000.], [-4500., 31000.]], dtype=dtype),
    torch.tensor([112.4, 95.7], dtype=dtype), slm_shape)
slm_to_camera = AffineRegistration(
    torch.tensor([[0.98, 0.035], [-0.025, 1.02]], dtype=dtype),
    torch.tensor([8.2, 7.8], dtype=dtype), camera_shape)
physical_to_camera = AffineRegistration(
    slm_to_camera.matrix @ truth.matrix,
    slm_to_camera.matrix @ truth.translation + slm_to_camera.translation,
    camera_shape)
camera_grid = Grid(camera_shape[1], camera_shape[0], 1., 1., dtype=dtype)
camera_pupil = physical_to_camera.resample_opd(pupil.transmission.to(dtype), grid).clamp(0, 1)
reference_amplitude = physical_to_camera.resample_opd(
    illuminated.complex_amplitude[0].real, grid)
camera = Detector(camera_grid, exposure_time=1e-3, photon_noise=False)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
show_map(ax[0], pupil.transmission, "Pupille / DM", "transmission")
show_map(ax[1], camera_pupil, "Relais sur la caméra", "transmission")
plt.tight_layout(); plt.show()

## 2. Mesurer la phase par quatre images d’interférence
Pour chaque poke ±, le DM applique une OPD réelle. Le relais transporte son champ complexe ; le détecteur FIATLUX mesure `|E + R exp(i δ)|²` pour δ = 0, π/2, π, 3π/2. La phase se reconstruit avec `atan2(Iπ/2 − I3π/2, I0 − Iπ)`.

Un faible bruit gaussien de lecture est ajouté avec une graine fixe. L’amplitude du poke reste sous π pour éviter le dépliage de phase. Nous soustrayons les phases push-pull et calculons le centroïde du signal mesuré, pas une position analytique.


In [ ]:
def measure_phase():
    field = dm.apply(illuminated)
    amplitude = field.complex_amplitude[0]
    relayed = (physical_to_camera.resample_opd(amplitude.real, grid)
               + 1j * physical_to_camera.resample_opd(amplitude.imag, grid))
    images = []
    for delta in (0., math.pi/2, math.pi, 3*math.pi/2):
        reference = reference_amplitude * complex(math.cos(delta), math.sin(delta))
        exposure = camera.acquire(Field((relayed + reference)[None], camera_grid, spectrum))
        noise = torch.randn(exposure.shape, generator=rng, dtype=dtype)
        images.append(exposure + 2e-4 * exposure.max() * noise)
    phase = torch.atan2(images[1] - images[3], images[0] - images[2])
    return phase, images

selected = select_registration_pokes(basis, pupil.transmission, count=12,
                                    minimum_illumination=0.99)
centers = square_zonal_centers(basis)
poke_opd = wavelength / 16
measured = []
responses = []
for index in selected.tolist():
    command = torch.zeros(basis.n_modes, dtype=dtype)
    command[index] = poke_opd
    dm.commands = command
    plus, exposures = measure_phase()
    dm.commands = -command
    minus, _ = measure_phase()
    response = torch.angle(torch.exp(1j * (plus - minus)))
    response = response * (camera_pupil > 0.99)
    measured.append(weighted_centroid(response, threshold_fraction=0.25))
    responses.append(response)
dm.flatten()
measured_camera = torch.stack(measured)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
show_map(ax[0], exposures[0], "Une image interférométrique", "électrons")
show_map(ax[1], responses[-1], "Réponse push-pull", "rad", cmap="RdBu_r")
plt.tight_layout(); plt.show()

## 3. Ajuster DM→caméra, puis composer caméra→SLM
Le fit ne reçoit que les centres commandés et les centroïdes mesurés. Les résidus sont en pixels caméra. La transformation utilisée par l’encodeur est ensuite exprimée en pixels SLM.


In [ ]:
fit_camera, residuals = AffineRegistration.fit(
    centers[selected], measured_camera, output_shape=camera_shape)
inverse_camera = torch.linalg.inv(slm_to_camera.matrix)
matched = AffineRegistration(
    inverse_camera @ fit_camera.matrix,
    inverse_camera @ (fit_camera.translation - slm_to_camera.translation), slm_shape)
rms_pixels = residuals.square().sum(-1).mean().sqrt()
# Tous les centres non utilisés au fit servent de validation géométrique indépendante.
held_out = torch.ones(basis.n_modes, dtype=torch.bool)
held_out[selected] = False
prediction_error = (matched.transform_points(centers[held_out])
                    - truth.transform_points(centers[held_out])).norm(dim=-1)
print(f"RMS des résidus caméra : {rms_pixels:.4f} px")
print(f"Erreur maximale hors fit : {prediction_error.max():.4f} px SLM")
print("Matrice ajustée (px/m) :", matched.matrix)
print("Translation ajustée (px) :", matched.translation)
assert rms_pixels < 0.4
assert prediction_error.max() < 0.7
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(*measured_camera.T, label="mesures", marker="+")
ax[0].scatter(*fit_camera.transform_points(centers[selected]).T,
              label="fit", facecolors="none", edgecolors="tab:orange")
ax[0].set(xlabel="u caméra (px)", ylabel="v caméra (px)", title="Correspondances")
ax[0].axis("equal"); ax[0].legend()
ax[1].quiver(*measured_camera.T, *residuals.T, angles="xy", scale_units="xy", scale=1)
ax[1].set(xlabel="u caméra (px)", ylabel="v caméra (px)", title="Résidus (échelle réelle)")
plt.tight_layout(); plt.show()

## 4. Calibration phase/commande et validation optique du matching
Une LUT synthétique non linéaire couvre 0 à 2π à la longueur d’onde du banc. L’encodeur rééchantillonne **l’OPD continue avant le modulo et la quantification 8 bits**. Le simulateur SLM inverse la même loi physique pour reconstruire la phase effectivement appliquée, puis la ramène sur la grille pupille avec la géométrie vraie.

On compare un SLM mal aligné, le SLM après matching et l’OPD idéale. Le motif est lisse pour séparer l’erreur de registration de l’interpolation des bords de modes carrés.


In [ ]:
levels = torch.linspace(0, 255, 256, dtype=dtype)
phase_lut = 2 * torch.pi * (levels / 255).pow(1.15)
calibration = SLMCalibration(wavelength, levels, phase_lut)
encoder = SLMEncoder(calibration, matched)
x, y = grid.meshgrid()
target_opd = 180e-9 * (0.6 * torch.sin(2*torch.pi*x/3e-3)
                      + 0.4 * torch.cos(2*torch.pi*y/2.5e-3))
frame = encoder.encode_opd(target_opd, grid)
# This inverse sensor model is local to the notebook, not a vendor SDK.
def applied_phase(frame):
    panel_phase = phase_lut[frame.long()]
    uv = truth.transform_points(torch.stack((x, y), dim=-1))
    sample = torch.stack((2*uv[..., 0]/(slm_shape[1]-1)-1,
                          2*uv[..., 1]/(slm_shape[0]-1)-1), dim=-1)
    # Interpolate phasors, never wrapped phase angles across a 2pi discontinuity.
    phasor = torch.exp(1j * panel_phase)
    real = F.grid_sample(phasor.real[None, None], sample[None], align_corners=True)[0, 0]
    imag = F.grid_sample(phasor.imag[None, None], sample[None], align_corners=True)[0, 0]
    return torch.atan2(imag, real)

bad = AffineRegistration(torch.tensor([[33000., 0.], [0., 33000.]], dtype=dtype),
                         torch.tensor([102., 103.], dtype=dtype), slm_shape)
bad_frame = SLMEncoder(calibration, bad).encode_opd(target_opd, grid)
ideal_phase = 2 * torch.pi * target_opd / wavelength
matched_phase, bad_phase = applied_phase(frame), applied_phase(bad_frame)
support = pupil.transmission.bool()
def phase_rms(phase):
    error = torch.angle(torch.exp(1j * (phase - ideal_phase)))[support]
    return error.square().mean().sqrt()
print(f"RMS phase avant : {phase_rms(bad_phase):.5f} rad")
print(f"RMS phase après : {phase_rms(matched_phase):.5f} rad")
assert frame.dtype == torch.uint8 and tuple(frame.shape) == slm_shape
assert phase_rms(matched_phase) < 0.04
assert phase_rms(matched_phase) < phase_rms(bad_phase) / 3
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
show_map(ax[0], target_opd*1e9, "OPD cible", "nm", cmap="RdBu_r")
show_map(ax[1], frame, "Image SLM après matching", "niveau 8 bits", cmap="gray")
error = torch.angle(torch.exp(1j*(matched_phase-ideal_phase))) * pupil.transmission
show_map(ax[2], error, "Erreur de phase dans la pupille", "rad", cmap="RdBu_r")
plt.tight_layout(); plt.show()

In [ ]:
lens = FFTPropagator(focal_length=0.2)
def focal_image(phase):
    slm_field = Field(illuminated.complex_amplitude * torch.exp(1j*phase), grid, spectrum)
    focal_field = lens.apply(slm_field)
    detector = Detector(focal_field.grid, exposure_time=1e-3, photon_noise=False)
    return detector.acquire(focal_field)
ideal_psf = focal_image(ideal_phase)
matched_psf = focal_image(matched_phase)
bad_psf = focal_image(bad_phase)
relative_psf_error = (matched_psf - ideal_psf).norm() / ideal_psf.norm()
print(f"Erreur relative L2 de la PSF : {relative_psf_error:.5f}")
assert relative_psf_error < 0.06
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for axis, image, title in zip(ax, (ideal_psf, bad_psf, matched_psf),
                             ("PSF idéale", "Avant matching", "Après matching")):
    show_map(axis, torch.log10(image.clamp_min(ideal_psf.max()*1e-7) / ideal_psf.max()),
             title, "log10(I / Imax idéale)", vmin=-7, vmax=0, cmap="inferno")
    axis.set_xlim(45, 83); axis.set_ylim(45, 83)
plt.tight_layout(); plt.show()

## 5. Sauvegarder et relire la calibration
Le fichier inclut la convention, la longueur d’onde, la LUT et la registration dans le repère **SLM**, pas caméra. Sur un banc réel, ajouter date, polarisation, numéro du panneau et conditions de mesure. Aucun fichier n’est écrit automatiquement : décommenter la ligne indiquée pour sauvegarder.


In [ ]:
payload = dict(convention="uv = xy @ matrix.T + translation; xy in metres; uv in SLM pixels",
    wavelength_m=wavelength, bit_depth=calibration.bit_depth,
    matrix=matched.matrix.tolist(), translation=matched.translation.tolist(),
    output_shape=list(slm_shape), drive_levels=levels.tolist(), phase_radians=phase_lut.tolist())
import json
serialized = json.dumps(payload, indent=2)
# Path("slm_calibration.json").write_text(serialized, encoding="utf-8")
loaded = json.loads(serialized)
restored = SLMEncoder(
    SLMCalibration(loaded["wavelength_m"], torch.tensor(loaded["drive_levels"], dtype=dtype),
                   torch.tensor(loaded["phase_radians"], dtype=dtype), loaded["bit_depth"]),
    AffineRegistration(torch.tensor(loaded["matrix"], dtype=dtype),
                       torch.tensor(loaded["translation"], dtype=dtype),
                       tuple(loaded["output_shape"])))
assert torch.equal(restored.encode_opd(target_opd, grid), frame)
print("Calibration relue : image de commande identique.")

## Portée de la démonstration
Les acquisitions sont produites par un champ optique et un détecteur FIATLUX ; aucune mesure n’est remplacée par une position théorique. La caméra observe un **DM de référence en amont**, qui donne les points physiques nécessaires au matching ; cela évite de supposer déjà connue la transformation recherchée pour envoyer les premiers pokes SLM.

La calibration SLM→caméra et la LUT sont fournies au départ. Leur acquisition réelle, les aberrations du relais, les défauts du panneau, la stabilité de la référence et le dépliage de phase ne sont pas démontrés ici. Les points hors fit vérifient la géométrie synthétique ; pour une validation expérimentale, acquérir des pokes supplémentaires indépendants. Une transformation affine ne corrige pas les distorsions non affines.
